In [1]:
from datasets import load_dataset 

imdb = load_dataset("imdb")

c:\Users\sdsc\Desktop\proj\fine-tune-sdsc4070\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer

MODEL = "distilbert/distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

In [3]:
tokenized_imdb = imdb.map(tokenize, batched=True)
tokenized_imdb

Map: 100%|██████████| 50000/50000 [00:25<00:00, 1938.58 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})

In [4]:
import numpy as np
from transformers import DataCollatorWithPadding
import evaluate

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy = evaluate.load("accuracy")

def compute_scores(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [5]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2, id2label=id2label, label2id=label2id
)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 5654.61it/s]
GPT2ForSequenceClassification LOAD REPORT from: distilbert/distilgpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
import torch
from transformers import TrainerCallback

def gpu_memory_stats():
    if not torch.cuda.is_available():
        return {}

    device = torch.cuda.current_device()
    gib = 1024 ** 3
    total = torch.cuda.get_device_properties(device).total_memory / gib

    return {
        "gpu_mem_allocated_gb": round(torch.cuda.memory_allocated(device) / gib, 3),
        "gpu_mem_reserved_gb": round(torch.cuda.memory_reserved(device) / gib, 3),
        "gpu_mem_peak_allocated_gb": round(torch.cuda.max_memory_allocated(device) / gib, 3),
        "gpu_mem_total_gb": round(total, 3),
    }

class GPUMemoryCallback(TrainerCallback):
    def on_train_begin(self, _args, _state, _control, **_kwargs):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def on_log(self, _args, state, _control, logs=None, **_kwargs):
        if logs is None or not torch.cuda.is_available():
            return

        stats = gpu_memory_stats()
        logs.update(stats)
        print(
            f"step={state.global_step} "
            f"allocated={stats['gpu_mem_allocated_gb']:.3f} GB "
            f"reserved={stats['gpu_mem_reserved_gb']:.3f} GB "
            f"peak={stats['gpu_mem_peak_allocated_gb']:.3f} GB"
        )

gpu_memory_stats()

{'gpu_mem_allocated_gb': 0.0,
 'gpu_mem_reserved_gb': 0.0,
 'gpu_mem_peak_allocated_gb': 0.0,
 'gpu_mem_total_gb': 7.996}

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    skip_memory_metrics=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_scores,
    callbacks=[GPUMemoryCallback()],
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Epoch,Training Loss,Validation Loss
